# FIT3182 Assignment 2 — Task 1: MongoDB Data Model

**Student IDs:** 34080678 | 33590982

This notebook implements **Task 1** of Assignment 2, covering the design and creation of MongoDB collections for the traffic speed camera system.

- **Task 1.1** — Collection Design: Vehicle, Camera, and Violation collections (schema, sample document, indexes, shard key, retention policy)
- **Task 1.2** — Collection Relationships: embed vs. reference justification, read/write patterns, trade-offs

**Dataset overview:**

| File | Description |
|------|-------------|
| `vehicle.csv` | Registered vehicles (plate, owner, type) |
| `camera.csv` | Speed camera metadata (location, speed limit) |
| `camera_event_A/B/C.csv` | Real-time speed reading events from 3 producers |
| `camera_event_historic.csv` | Historic speed violations |

In [8]:
# ── Installation & Imports ──────────────────────────────────────────────────
# pymongo is pre-installed in this environment.
# To reinstall: python -m pip install pymongo --target <venv>/Lib/site-packages

import pymongo
from pymongo import MongoClient, ASCENDING, DESCENDING
import pandas as pd
import json
from datetime import datetime

print(f"pymongo version: {pymongo.__version__}")
print("All required libraries imported successfully.")

pymongo version: 4.17.0
All required libraries imported successfully.


## MongoDB Connection

Connect to the MongoDB instance running in Docker on port **27017**.  
The database `fit3182_db` is used throughout this assignment.

In [9]:
# ── MongoDB Connection ───────────────────────────────────────────────────────
# MongoDB is running in Docker container 'fervent_hamilton' on port 27017.
# Change MONGO_HOST if your Docker host IP differs (e.g. '192.168.x.x').
MONGO_HOST = "localhost"
MONGO_PORT = 27017
DB_NAME    = "fit3182_db"

client = MongoClient(MONGO_HOST, MONGO_PORT, serverSelectionTimeoutMS=5000)

# Verify connectivity
try:
    client.admin.command("ping")
    print(f"Connected to MongoDB at {MONGO_HOST}:{MONGO_PORT}")
except Exception as e:
    print(f"Connection failed: {e}")
    raise

db = client[DB_NAME]
print(f"Using database: '{DB_NAME}'")

Connected to MongoDB at localhost:27017
Using database: 'fit3182_db'


---
## Task 1.1 — Collection Design

Four collections are designed for this system:

| Collection | Purpose |
|---|---|
| `vehicle` | Registered vehicle and owner information |
| `camera` | Speed camera locations and speed limits |
| `violation` | Detected speeding violations (streaming + historic) |
| `camera_event` | Raw speed-reading events captured by cameras |

---
### Collection 1: `vehicle`

**Description:**  
Stores registered vehicle data including the owner's details and vehicle type. The `car_plate` field uniquely identifies each vehicle and acts as a natural primary key and the foreign reference used in both `violation` and `camera_event` collections.

**Document Schema:**

```json
{
  "car_plate":         "string  — unique licence plate (PK)",
  "owner_name":        "string  — full name of registered owner",
  "owner_addr":        "string  — residential address",
  "vehicle_type":      "string  — e.g. Sedan, SUV, Coupe, Hatchback",
  "registration_date": "Date    — ISO 8601 date of registration"
}
```

**Indexes:**
| Field | Type | Reason |
|---|---|---|
| `car_plate` | Unique ascending | Primary lookup key; enforces uniqueness |
| `vehicle_type` | Ascending | Supports analytics queries filtered by vehicle type |

**Shard Key:** `car_plate` — high cardinality, evenly distributed, used in all cross-collection joins.

**Retention Policy:** Records are kept indefinitely; vehicles deregistered > 7 years ago may be archived to cold storage.

In [12]:
# ── Collection 1: vehicle ────────────────────────────────────────────────────
# Drop and recreate for a clean run
db.drop_collection("vehicle")

# Create collection with JSON Schema validation
db.create_collection("vehicle", validator={
    "$jsonSchema": {
        "bsonType": "object",
        "required": ["car_plate", "owner_name", "vehicle_type", "registration_date"],
        "properties": {
            "car_plate":         {"bsonType": "string",   "description": "Unique licence plate"},
            "owner_name":        {"bsonType": "string",   "description": "Registered owner name"},
            "owner_addr":        {"bsonType": "string",   "description": "Owner address"},
            "vehicle_type":      {"bsonType": "string",   "enum": ["Sedan","SUV","Coupe","Hatchback","Van","Truck"]},
            "registration_date": {"bsonType": "date",     "description": "ISO date of registration"},
        }
    }
})

# Create indexes
vehicle_col = db["vehicle"]
vehicle_col.create_index([("car_plate", ASCENDING)], unique=True, name="idx_car_plate_unique")
vehicle_col.create_index([("vehicle_type", ASCENDING)],            name="idx_vehicle_type")
print("Indexes created:", list(vehicle_col.index_information().keys()))

# Load data from CSV
# Note: source CSV has a typo 'vechicle_type' — corrected to 'vehicle_type' on load
df_vehicle = pd.read_csv("../../data/vehicle.csv")
df_vehicle.rename(columns={"vechicle_type": "vehicle_type"}, inplace=True)
df_vehicle["registration_date"] = pd.to_datetime(df_vehicle["registration_date"])
# Deduplicate on car_plate, keeping first occurrence
df_vehicle.drop_duplicates(subset="car_plate", keep="first", inplace=True)

records = df_vehicle.to_dict(orient="records")
result = vehicle_col.insert_many(records)
print(f"Inserted {len(result.inserted_ids)} vehicle documents (after deduplication)")

# Display a sample document
sample = vehicle_col.find_one({}, {"_id": 0})
print("\nSample document:")
print(json.dumps({k: str(v) for k, v in sample.items()}, indent=2))

Indexes created: ['_id_', 'idx_car_plate_unique', 'idx_vehicle_type']
Inserted 9844 vehicle documents (after deduplication)

Sample document:
{
  "car_plate": "FT 02",
  "owner_name": "Goh Mei Wei",
  "owner_addr": "943 Jalan Bukit Mawar, Kuala Lumpur",
  "vehicle_type": "Coupe",
  "registration_date": "2006-08-22 03:18:00"
}


---
### Collection 2: `camera`

**Description:**  
Stores speed camera metadata including GPS coordinates, road position, and the enforced speed limit. This is a small, **read-heavy, rarely updated** reference collection. It is referenced by both `camera_event` and `violation` to avoid duplicating speed-limit data.

**Document Schema:**

```json
{
  "camera_id":    "int     — unique camera identifier (PK)",
  "latitude":     "double  — GPS latitude",
  "longitude":    "double  — GPS longitude",
  "position":     "double  — road position marker (km)",
  "speed_limit":  "int     — enforced speed limit (km/h)"
}
```

**Indexes:**
| Field | Type | Reason |
|---|---|---|
| `camera_id` | Unique ascending | Primary lookup; joins from events and violations |
| `speed_limit` | Ascending | Supports queries filtering cameras by speed zone |

**Shard Key:** `camera_id` — monotonically increasing integer; suitable for range-based sharding.

**Retention Policy:** Permanent — camera records represent physical infrastructure. Decommissioned cameras should be marked inactive rather than deleted (to preserve historical violation context).

In [13]:
# ── Collection 2: camera ─────────────────────────────────────────────────────
db.drop_collection("camera")

db.create_collection("camera", validator={
    "$jsonSchema": {
        "bsonType": "object",
        "required": ["camera_id", "latitude", "longitude", "speed_limit"],
        "properties": {
            "camera_id":   {"bsonType": "int",    "description": "Unique camera ID"},
            "latitude":    {"bsonType": "double", "description": "GPS latitude"},
            "longitude":   {"bsonType": "double", "description": "GPS longitude"},
            "position":    {"bsonType": "double", "description": "Road position marker (km)"},
            "speed_limit": {"bsonType": "int",    "description": "Enforced speed limit (km/h)"},
        }
    }
})

camera_col = db["camera"]
camera_col.create_index([("camera_id", ASCENDING)], unique=True, name="idx_camera_id_unique")
camera_col.create_index([("speed_limit", ASCENDING)],             name="idx_speed_limit")
print("Indexes created:", list(camera_col.index_information().keys()))

# Load data from CSV
df_camera = pd.read_csv("../../data/camera.csv")
df_camera["camera_id"]   = df_camera["camera_id"].astype(int)
df_camera["speed_limit"] = df_camera["speed_limit"].astype(int)

records = df_camera.to_dict(orient="records")
result = camera_col.insert_many(records)
print(f"Inserted {len(result.inserted_ids)} camera documents")

sample = camera_col.find_one({}, {"_id": 0})
print("\nSample document:")
print(json.dumps(sample, indent=2))

Indexes created: ['_id_', 'idx_camera_id_unique', 'idx_speed_limit']
Inserted 3 camera documents

Sample document:
{
  "camera_id": 1,
  "latitude": 2.157730731,
  "longitude": 102.6601002,
  "position": 152.5,
  "speed_limit": 110
}


---
### Collection 3: `violation`

**Description:**  
Stores confirmed speeding violations detected either from the real-time streaming pipeline or loaded from historic records. A violation is produced when a vehicle's average speed between two cameras exceeds the speed limit. This collection is the primary **write target** for the streaming sink and the primary **read source** for dashboards and reporting.

**Document Schema:**

```json
{
  "violation_id":    "string  — UUID (PK)",
  "car_plate":       "string  — reference to vehicle.car_plate",
  "camera_id_start": "int     — reference to camera.camera_id (entry point)",
  "camera_id_end":   "int     — reference to camera.camera_id (exit point)",
  "timestamp_start": "Date    — event timestamp at entry camera",
  "timestamp_end":   "Date    — event timestamp at exit camera",
  "speed_reading":   "double  — computed average speed (km/h)",
  "speed_limit":     "int     — embedded speed limit at time of violation",
  "source":          "string  — 'streaming' or 'historic'"
}
```

**Indexes:**
| Field | Type | Reason |
|---|---|---|
| `violation_id` | Unique ascending | Deduplication; primary key |
| `car_plate` | Ascending | Fast lookup of all violations for a vehicle |
| `camera_id_start` | Ascending | Filter violations by camera zone |
| `timestamp_start` | Descending | Time-range queries, dashboards |

**Shard Key:** `car_plate` — consistent with `vehicle` collection; enables co-location of a vehicle's documents across shards.

**Retention Policy:** Violations are legally significant records. Retain indefinitely. Consider a TTL index on a `archived_at` field (e.g. 7 years) for regulatory purging if required.

In [15]:
# ── Collection 3: violation ──────────────────────────────────────────────────
db.drop_collection("violation")

db.create_collection("violation", validator={
    "$jsonSchema": {
        "bsonType": "object",
        "required": ["violation_id", "car_plate", "camera_id_start", "camera_id_end",
                     "timestamp_start", "speed_reading"],
        "properties": {
            "violation_id":    {"bsonType": "string", "description": "UUID primary key"},
            "car_plate":       {"bsonType": "string", "description": "Ref: vehicle.car_plate"},
            "camera_id_start": {"bsonType": "int",    "description": "Ref: camera.camera_id"},
            "camera_id_end":   {"bsonType": "int",    "description": "Ref: camera.camera_id"},
            "timestamp_start": {"bsonType": "date",   "description": "Entry camera timestamp"},
            "timestamp_end":   {"bsonType": "date",   "description": "Exit camera timestamp"},
            "speed_reading":   {"bsonType": "double", "description": "Computed avg speed (km/h)"},
            "speed_limit":     {"bsonType": "int",    "description": "Speed limit at violation time"},
            "source":          {"bsonType": "string", "enum": ["streaming", "historic"]},
        }
    }
})

violation_col = db["violation"]
violation_col.create_index([("violation_id", ASCENDING)],    unique=True, name="idx_violation_id_unique")
violation_col.create_index([("car_plate", ASCENDING)],                    name="idx_viol_car_plate")
violation_col.create_index([("camera_id_start", ASCENDING)],              name="idx_viol_camera_start")
violation_col.create_index([("timestamp_start", DESCENDING)],             name="idx_viol_timestamp")
print("Indexes created:", list(violation_col.index_information().keys()))

# Load historic violations from CSV
# Timestamps have mixed precision (some with microseconds), use format='mixed'
df_hist = pd.read_csv("../../data/camera_event_historic.csv")
df_hist["timestamp_start"] = pd.to_datetime(df_hist["timestamp_start"], format="mixed")
df_hist["timestamp_end"]   = pd.to_datetime(df_hist["timestamp_end"],   format="mixed")
df_hist["camera_id_start"] = df_hist["camera_id_start"].astype(int)
df_hist["camera_id_end"]   = df_hist["camera_id_end"].astype(int)
df_hist["source"]          = "historic"

# Join with camera to embed speed_limit at time of violation (camera_id_start)
camera_speed = {
    doc["camera_id"]: doc["speed_limit"]
    for doc in camera_col.find({}, {"_id": 0, "camera_id": 1, "speed_limit": 1})
}
df_hist["speed_limit"] = df_hist["camera_id_start"].map(camera_speed).fillna(0).astype(int)

records = df_hist.to_dict(orient="records")
result = violation_col.insert_many(records)
print(f"Inserted {len(result.inserted_ids)} historic violation documents")

sample = violation_col.find_one({}, {"_id": 0})
print("\nSample document:")
print(json.dumps({k: str(v) for k, v in sample.items()}, indent=2))

Indexes created: ['_id_', 'idx_violation_id_unique', 'idx_viol_car_plate', 'idx_viol_camera_start', 'idx_viol_timestamp']
Inserted 50000 historic violation documents

Sample document:
{
  "violation_id": "0ff38c74-7ee6-41cd-bc26-3a6060604a02",
  "car_plate": "YE 6517",
  "camera_id_start": "1",
  "camera_id_end": "2",
  "timestamp_start": "2018-11-14 08:30:11",
  "timestamp_end": "2018-11-14 08:30:35.821000",
  "speed_reading": "145.0",
  "source": "historic",
  "speed_limit": "110"
}


---
### Collection 4: `camera_event`

**Description:**  
Stores raw speed-reading events as they arrive from the three camera producers (A, B, C) via Kafka. This collection acts as a **staging / audit log** for the streaming pipeline. Events here are joined in Spark Structured Streaming to produce violations. Not every event is a violation — this collection captures all readings.

**Document Schema:**

```json
{
  "event_id":   "string  — UUID (PK)",
  "batch_id":   "int     — Kafka batch number",
  "car_plate":  "string  — reference to vehicle.car_plate",
  "camera_id":  "int     — reference to camera.camera_id",
  "timestamp":  "Date    — event capture time",
  "speed_reading": "double — instantaneous speed reading (km/h)"
}
```

**Indexes:**
| Field | Type | Reason |
|---|---|---|
| `event_id` | Unique ascending | Deduplication |
| `car_plate, timestamp` | Compound ascending | Join key for streaming window |
| `camera_id, timestamp` | Compound ascending | Per-camera time-ordered queries |

**Shard Key:** `camera_id` — events are naturally partitioned by camera; co-locates all events for the same camera on one shard.

**Retention Policy:** Short-term operational data. Apply a TTL index of **30 days** — events are only needed until the streaming pipeline processes them into violations.

In [17]:
# ── Collection 4: camera_event ───────────────────────────────────────────────
db.drop_collection("camera_event")

db.create_collection("camera_event", validator={
    "$jsonSchema": {
        "bsonType": "object",
        "required": ["event_id", "car_plate", "camera_id", "timestamp", "speed_reading"],
        "properties": {
            "event_id":      {"bsonType": "string", "description": "UUID primary key"},
            "batch_id":      {"bsonType": "int",    "description": "Kafka batch number"},
            "car_plate":     {"bsonType": "string", "description": "Ref: vehicle.car_plate"},
            "camera_id":     {"bsonType": "int",    "description": "Ref: camera.camera_id"},
            "timestamp":     {"bsonType": "date",   "description": "Event capture time"},
            "speed_reading": {"bsonType": "double", "description": "Speed reading (km/h)"},
        }
    }
})

event_col = db["camera_event"]
event_col.create_index([("event_id", ASCENDING)],                          unique=True, name="idx_event_id_unique")
event_col.create_index([("car_plate", ASCENDING), ("timestamp", ASCENDING)],            name="idx_plate_time")
event_col.create_index([("camera_id", ASCENDING), ("timestamp", ASCENDING)],            name="idx_camera_time")
# TTL index: auto-delete documents 30 days after capture
event_col.create_index([("timestamp", ASCENDING)], expireAfterSeconds=30*24*3600,        name="ttl_30days")
print("Indexes created:", list(event_col.index_information().keys()))

# Load all three producer event files
# Timestamps have mixed precision — use format='mixed'
dfs = []
for producer in ["A", "B", "C"]:
    df = pd.read_csv(f"../../data/camera_event_{producer}.csv")
    df["producer"] = producer
    dfs.append(df)

df_events = pd.concat(dfs, ignore_index=True)
df_events["timestamp"]  = pd.to_datetime(df_events["timestamp"], format="mixed")
df_events["camera_id"]  = df_events["camera_id"].astype(int)
df_events["batch_id"]   = df_events["batch_id"].astype(int)

records = df_events.to_dict(orient="records")
result = event_col.insert_many(records)
print(f"Inserted {len(result.inserted_ids)} camera_event documents across producers A, B, C")

sample = event_col.find_one({}, {"_id": 0})
print("\nSample document:")
print(json.dumps({k: str(v) for k, v in sample.items()}, indent=2))

Indexes created: ['_id_', 'idx_event_id_unique', 'idx_plate_time', 'idx_camera_time', 'ttl_30days']
Inserted 1669020 camera_event documents across producers A, B, C

Sample document:
{
  "event_id": "cc691ba1-64e2-4b28-8227-f9389533e139",
  "batch_id": "2773",
  "car_plate": "WCM 6019",
  "camera_id": "1",
  "timestamp": "2024-01-23 03:19:57",
  "speed_reading": "150.1",
  "producer": "A"
}


---
## Task 1.2 — Collection Relationships

### Relationship Diagram

```
vehicle (car_plate PK)
    │
    ├──── referenced by ──── camera_event.car_plate
    └──── referenced by ──── violation.car_plate

camera (camera_id PK)
    │
    ├──── referenced by ──── camera_event.camera_id
    ├──── referenced by ──── violation.camera_id_start
    └──── referenced by ──── violation.camera_id_end
```

---

### Design Decision: Reference vs Embed

All relationships use **referencing (normalisation)** rather than embedding. The justification for each link is discussed below.

---

#### `camera_event` → `vehicle` (via `car_plate`)

**Approach: Reference**

- `vehicle` records are large (owner name, address, type) relative to the high-volume event stream.
- Events arrive at **millions per day** from three producers; embedding vehicle details into every event would multiply storage costs significantly.
- Vehicle data **rarely changes** but is **read infrequently** from the events collection — dashboards join violation → vehicle, not event → vehicle.
- A reference keeps events lean and write-fast, which is critical for the streaming sink.

---

#### `camera_event` → `camera` (via `camera_id`)

**Approach: Reference (with `speed_limit` selectively embedded in `violation`)**

- There are only 3 cameras — the `camera` collection is tiny and fits entirely in memory.
- Referencing avoids duplicating speed-limit values across thousands of events.
- **Exception:** `violation` embeds `speed_limit` at write time. Since a violation is a legal record, the enforced limit must be immutable even if the camera's limit is later updated. This is a deliberate **embed for immutability** rather than a join.

---

#### `violation` → `vehicle` (via `car_plate`)

**Approach: Reference**

- Violations are **written once and read many times** (reports, dashboards, enforcement).
- Owner details can change (address update, ownership transfer). Referencing ensures queries always return current owner information without backfilling violation documents.
- The trade-off is a two-collection lookup (`violation` → `vehicle`), but this is acceptable since violation queries are typically filtered first by `car_plate` or `timestamp`, and the vehicle lookup is a single-key point query on an indexed field.

---

### Consistency Considerations

MongoDB does not enforce foreign-key constraints. Application-level consistency is maintained by:

1. **Insert order:** `vehicle` and `camera` documents are loaded before any `camera_event` or `violation` documents.
2. **Validation schemas** on each collection reject documents with missing required fields.
3. **Unique indexes** on `car_plate`, `camera_id`, `event_id`, and `violation_id` prevent duplicate inserts from stream retries.

---

### Summary Table

| Relationship | Strategy | Reason |
|---|---|---|
| `camera_event.car_plate` → `vehicle` | Reference | High write volume; vehicle data rarely needed from raw events |
| `camera_event.camera_id` → `camera` | Reference | Camera collection tiny; avoids value duplication |
| `violation.car_plate` → `vehicle` | Reference | Owner data changes; reference ensures freshness |
| `violation.camera_id_start/end` → `camera` | Reference | Camera list is static reference data |
| `violation.speed_limit` (embedded) | Embed | Legal immutability — captured at violation time |

In [18]:
# ── Database Summary Verification ────────────────────────────────────────────
print(f"Collections in '{DB_NAME}':")
print("-" * 45)
for col_name in sorted(db.list_collection_names()):
    col = db[col_name]
    count = col.count_documents({})
    indexes = list(col.index_information().keys())
    print(f"  {col_name:<18} {count:>6} documents   indexes: {indexes}")

print("\nTask 1 setup complete.")

Collections in 'fit3182_db':
---------------------------------------------
  camera                  3 documents   indexes: ['_id_', 'idx_camera_id_unique', 'idx_speed_limit']
  camera_event       1230680 documents   indexes: ['_id_', 'idx_event_id_unique', 'idx_plate_time', 'idx_camera_time', 'ttl_30days']
  vehicle              9844 documents   indexes: ['_id_', 'idx_car_plate_unique', 'idx_vehicle_type']
  violation           50000 documents   indexes: ['_id_', 'idx_violation_id_unique', 'idx_viol_car_plate', 'idx_viol_camera_start', 'idx_viol_timestamp']

Task 1 setup complete.
